In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm  # version Jupyter-friendly
# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)

# --- paramètres de base ---
folder = VIDEO_DIR
solution = "pose"      # 'pose', 'hands', 'face', ou 'holistic'
stride = 1             # 1 = chaque frame ; 2 = une sur deux
show_video = True      # affiche la vidéo pendant le traitement

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
mp_holistic = mp.solutions.holistic

# noms des marqueurs pour MediaPipe Pose
POSE_LANDMARKS = [lm.name for lm in mp_pose.PoseLandmark]

def process_video(video_path, solution="pose", stride=1, show=False):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    rows = []

    # choisir le modèle MediaPipe
    if solution == "pose":
        model = mp_pose.Pose(model_complexity=1)
    elif solution == "hands":
        model = mp_hands.Hands(max_num_hands=2)
    elif solution == "face":
        model = mp_face.FaceMesh(refine_landmarks=True)
    elif solution == "holistic":
        model = mp_holistic.Holistic(model_complexity=1)
    else:
        raise ValueError("solution inconnue")

    with model as m:
        for i in tqdm(range(nframes), desc=video_path.name):
            ok, frame = cap.read()
            if not ok:
                break
            if i % stride != 0:
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = m.process(rgb)

            # affichage vidéo avec landmarks
            if show:
                annotated = frame.copy()
                if solution == "pose" and res.pose_landmarks:
                    mp_drawing.draw_landmarks(annotated, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                elif solution == "holistic":
                    if res.pose_landmarks:
                        mp_drawing.draw_landmarks(annotated, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                    if res.left_hand_landmarks:
                        mp_drawing.draw_landmarks(annotated, res.left_hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    if res.right_hand_landmarks:
                        mp_drawing.draw_landmarks(annotated, res.right_hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    if res.face_landmarks:
                        mp_drawing.draw_landmarks(annotated, res.face_landmarks, mp_face.FACEMESH_TESSELATION)
                cv2.imshow("Processing", cv2.resize(annotated, (640, 480)))
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            # enregistrement des coordonnées du corps
            if hasattr(res, "pose_landmarks") and res.pose_landmarks:
                data = {"frame": i, "t_ms": (i / fps) * 1000}
                for j, lm in enumerate(res.pose_landmarks.landmark):
                    name = POSE_LANDMARKS[j]
                    data[f"{name}_x"] = lm.x
                    data[f"{name}_y"] = lm.y
                    data[f"{name}_z"] = lm.z
                    data[f"{name}_v"] = getattr(lm, "visibility", 0)
                rows.append(data)

    cap.release()
    cv2.destroyAllWindows()
    df = pd.DataFrame(rows)
    out = video_path.with_name(f"{video_path.stem}_{solution}.xlsx")
    df.to_excel(out, index=False)
    print(f"✅ Sauvegardé : {out}")

# --- boucle sur toutes les vidéos ---
videos = [v for v in folder.rglob("*") if v.suffix.lower() in [".mp4", ".mov", ".avi"]]
print(f"{len(videos)} vidéo(s) trouvée(s).")
for v in videos:
    process_video(v, solution=solution, stride=stride, show=show_video)

48 vidéo(s) trouvée(s).


I0000 00:00:1776201601.373058 1430001 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M2 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776201601.466740 1430211 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776201601.485108 1430214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


SEATED.mp4:   0%|          | 0/4506 [00:00<?, ?it/s]

W0000 00:00:1776201601.902048 1430216 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
2026-04-14 23:20:02.324 Python[34896:1430001] +[IMKClient subclass]: chose IMKClient_Modern
2026-04-14 23:20:02.324 Python[34896:1430001] +[IMKInputSession subclass]: chose IMKInputSession_Modern


✅ Sauvegardé : /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D15/P2/SEATED/SEATED_pose.xlsx


...

✅ Sauvegardé : /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D17/P1/SEMI/SEMI_pose.xlsx


I0000 00:00:1776212535.713530 1430001 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M2 Pro


STANDING.mp4:   0%|          | 0/4508 [00:00<?, ?it/s]

W0000 00:00:1776212535.798933 1554730 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776212535.815202 1554730 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✅ Sauvegardé : /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D17/P1/STANDING/STANDING_pose.xlsx


In [2]:
print(folder)
print(list(folder.iterdir()))

/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video
[PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Summary_Motion_Nose_Wrists_equalizedFrames.xlsx'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D15'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D14'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D13'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/.DS_Store'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Summary_Motion_Nose_Wrists_equalizedFrames.csv'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/mediapipe_QDM_filtered_and_SW.xlsx'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D16'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D18'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Shoulder_norm.xlsx'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D20'), PosixPath('/Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/D